<a href="https://colab.research.google.com/github/vr11-ai/Lightening-Prediction-Model/blob/main/Lightning_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from google.colab import drive
drive.mount('/content/drive')

# 1. Load Data
print("Loading data...")
df=pd.read_csv('/content/drive/MyDrive/datasets/Prayagraj_labelled.csv')
print("Data loaded!\n")

# 2. Feature Engineering (Temporal)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
X = df.drop(columns=['lightning', 'Location', 'timestamp'])
y = df['lightning']

# 3. Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Data Augmentation: "Adding Values" to prevent overfitting
# We create 500 synthetic lightning samples with 1% random noise
X_min = X_train[y_train == 1]
new_X = []
for _ in range(500):
    sample = X_min.iloc[np.random.randint(0, len(X_min))].values
    noise = np.random.normal(0, 0.01, size=sample.shape) # 1% noise
    new_X.append(sample + noise)

X_aug = pd.DataFrame(new_X, columns=X_train.columns)
y_aug = pd.Series([1] * 500)

# Combine and Downsample the majority class to 1,500 samples
df_train = pd.concat([X_train, y_train.rename('lightning')], axis=1)
df_maj = df_train[df_train.lightning == 0]
df_maj_down = resample(df_maj, replace=False, n_samples=1500, random_state=42)

# Final Training Set
X_train_bal = pd.concat([df_maj_down.drop('lightning', axis=1), X_aug])
y_train_bal = pd.concat([df_maj_down['lightning'], y_aug])

# 5. Scaling
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_bal)
X_test_final = scaler.transform(X_test)

# 6. Optimized Models with Regularization
# KNN (Higher k = smoother boundary)
knn = KNeighborsClassifier(n_neighbors=11).fit(X_train_final, y_train_bal)

# GBoost (Low depth = prevents overfitting)
gboost = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42).fit(X_train_final, y_train_bal)

# XGBoost (Strong Regularization)
xgb = XGBClassifier(
    n_estimators=50,
    max_depth=3,
    reg_lambda=1,   # L2 Regularization
    reg_alpha=0.1,  # L1 Regularization
    use_label_encoder=False,
    eval_metric='logloss'
).fit(X_train_final, y_train_bal)

# 7. Final Results
for name, model in [("KNN", knn), ("GBoost", gboost), ("XGBoost", xgb)]:
    preds = model.predict(X_test_final)
    print(f"\n--- {name} Results ---")
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.2%}")
    print(confusion_matrix(y_test, preds))

Mounted at /content/drive
Loading data...
Data loaded!



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:41:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- KNN Results ---
Balanced Accuracy: 74.12%
[[22874  2238]
 [    3     4]]

--- GBoost Results ---
Balanced Accuracy: 63.44%
[[24687   425]
 [    5     2]]

--- XGBoost Results ---
Balanced Accuracy: 56.71%
[[24897   215]
 [    6     1]]


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# 2. Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Data Augmentation: Adding "Noisy" values to prevent overfitting
# Generate 500 synthetic lightning samples with 1% random noise
X_min = X_train[y_train == 1]
new_X = []
for _ in range(500):
    sample = X_min.iloc[np.random.randint(0, len(X_min))].values
    noise = np.random.normal(0, 0.01, size=sample.shape)
    new_X.append(sample + noise)

X_aug = pd.DataFrame(new_X, columns=X_train.columns)
y_aug = pd.Series([1] * 500)

# 4. Balancing: Combine augmented data and downsample majority
df_maj = X_train[y_train == 0]
df_maj_down = resample(df_maj, replace=False, n_samples=1500, random_state=42)

X_train_final = pd.concat([df_maj_down, X_aug])
y_train_final = pd.concat([pd.Series([0]*1500), y_aug])

# 5. Scaling (Crucial for SVM and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test)

# 6. SVM Model with Regularization (C=0.5 helps prevent overfitting)
svm_model = SVC(kernel='rbf', C=0.5, probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train_final)

# 7. Evaluate and Compare
y_pred_svm = svm_model.predict(X_test_scaled)

print("--- SVM Results ---")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_svm):.2%}")
print(confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

--- SVM Results ---
Balanced Accuracy: 75.52%
[[23581  1531]
 [    3     4]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     25112
           1       0.00      0.57      0.01         7

    accuracy                           0.94     25119
   macro avg       0.50      0.76      0.49     25119
weighted avg       1.00      0.94      0.97     25119



In [ ]:
from sklearn.metrics import recall_score, confusion_matrix

# Assume models are already trained as 'knn', 'svm_model', 'gboost', and 'xgb'
# and X_test_scaled, y_test are available

models = {
    "KNN": knn,
    "SVM": svm_model,
    "GBoost": gboost,
    "XGBoost": xgb
}

print("--- Probability of Detection (POD) ---")
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)

    # Calculate POD (Recall for the positive class)
    pod = recall_score(y_test, y_pred)

    # Get True Positives and False Negatives from Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    print(f"{name}:")
    print(f"  POD: {pod:.2%}")
    print(f"  Detected: {tp} | Missed: {fn}")
    print("-" * 30)

--- Probability of Detection (POD) ---
KNN:
  POD: 57.14%
  Detected: 4 | Missed: 3
------------------------------
SVM:
  POD: 57.14%
  Detected: 4 | Missed: 3
------------------------------
GBoost:
  POD: 85.71%
  Detected: 6 | Missed: 1
------------------------------
XGBoost:
  POD: 71.43%
  Detected: 5 | Missed: 2
------------------------------


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# --- STEP 0: FIX ALL SEEDS FOR REPRODUCIBILITY ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 2. Feature Engineering
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
X = df.drop(columns=['lightning', 'Location', 'timestamp'])
y = df['lightning']

# 3. Stratified Split (Fixed seed)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# 4. Data Augmentation (Fixed seed via np.random.seed above)
X_min = X_train[y_train == 1]
new_X = []
for _ in range(500):
    # Picking samples and adding noise is now reproducible because of np.random.seed
    sample = X_min.iloc[np.random.randint(0, len(X_min))].values
    noise = np.random.normal(0, 0.01, size=sample.shape)
    new_X.append(sample + noise)

X_aug = pd.DataFrame(new_X, columns=X_train.columns)
y_aug = pd.Series([1] * 500)

# 5. Balancing
df_train = pd.concat([X_train, y_train.rename('lightning')], axis=1)
df_maj = df_train[df_train.lightning == 0]
df_maj_down = resample(df_maj, replace=False, n_samples=1500, random_state=RANDOM_SEED)

X_train_bal = pd.concat([df_maj_down.drop('lightning', axis=1), X_aug])
y_train_bal = pd.concat([df_maj_down['lightning'], y_aug])

# 6. Scaling
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_bal)
X_test_final = scaler.transform(X_test)

# 7. Initialize Models with Fixed Seeds
models = {
    "KNN": KNeighborsClassifier(n_neighbors=11), # KNN is usually deterministic
    "GBoost": GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_SEED),
    "SVM": SVC(kernel='rbf', C=0.5, probability=True, random_state=RANDOM_SEED),
    "XGBoost": XGBClassifier(n_estimators=50, max_depth=3, reg_lambda=1, random_state=RANDOM_SEED, eval_metric='logloss')
}

# 8. Train and Evaluate
print(f"{'Model':<10} | {'Bal. Accuracy':<15} | {'POD (Detection)':<15}")
print("-" * 50)

for name, model in models.items():
    model.fit(X_train_final, y_train_bal)
    preds = model.predict(X_test_final)

    acc = balanced_accuracy_score(y_test, preds)
    pod = recall_score(y_test, preds) # POD is Recall for the lightning class

    print(f"{name:<10} | {acc:<15.2%} | {pod:<15.2%}")

Model      | Bal. Accuracy   | POD (Detection)
--------------------------------------------------
KNN        | 74.10%          | 57.14%         
GBoost     | 56.37%          | 14.29%         
SVM        | 61.60%          | 28.57%         
XGBoost    | 56.68%          | 14.29%         
